# NCAA 疯狂三月 2026

由 2025 年冠军方案 (modeh7) 适配至 2026。  
相对原版的改动：

- 数据路径已更新为 2026 竞赛数据集  
- 提交文件：`SampleSubmissionStage1.csv`（Stage 2 在 bracket 确定后开放）  
- 已移除手动覆写（2026 对阵尚未确定）

### 流程概览

| 阶段 | 构建内容 | 核心思路 |
|-------|--------------|----------|
| **1. 加载** | 原始技术统计 DataFrame | 男篮+女篮通过 TeamID 范围统一 |
| **2. 准备** | 对称翻倍数据集 | 每场比赛录入两次（T1↔T2 互换） |
| **3. 基础特征** | 种子、种子差 | 遴选委员会专家知识 |
| **4. 中等特征** | 赛季场均技术统计 + 对手场均 | 队伍实力 + 防守质量代理 |
| **5. 复杂特征** | Elo 评级 | 赛季累计胜负动量 |
| **6. 最复杂特征** | GLM 队伍质量 | Bradley-Terry 式系数，用高斯 GLM 估计 |
| **7. 模型** | XGBoost 回归 → 分差 | 回归比二分类提供更丰富信号 |
| **8. 校准** | 样条：分差→概率 | 非参数，在 OOF 锦标赛结果上拟合 |
| **9. 提交** | LOSO 模型集成 | 所有留一赛季模型预测取平均 |

### 评估指标

Kaggle 使用 **Brier 分数**（预测概率与结果的均方误差）：

$$\text{Brier} = \frac{1}{N}\sum_{i=1}^{N}(p_i - y_i)^2$$

与 log-loss 不同，Brier 分数对过度自信的错误预测**不会**无限放大——每场比赛的最大惩罚为 1.0。这使得极端预测（例如 1 号种子对 16 号种子预测 0.99）比在 log-loss 下风险更小。尽管如此，我们仍将预测截断至 [0.01, 0.99] 并投入校准（第 8 节）以最小化总体误差。


基于 modeh7 原方案：
https://www.kaggle.com/code/modeh7/final-solution-ncaa-2025


## <<- 加载数据 ->>


## 1. 加载数据

我们加载男篮和女篮的**详细赛果**（技术统计）文件。

| 文件 | 内容 | 可追溯至 |
|------|----------|---------------|
| `MRegularSeasonDetailedResults` | 男篮常规赛每场完整技术统计 | 2003 |
| `WRegularSeasonDetailedResults` | 女篮常规赛每场完整技术统计 | 2010 |
| `M/WNCAATourneyDetailedResults` | 锦标赛完整技术统计 | 同上 |
| `M/WNCAATourneySeeds` | 每支锦标赛队伍的种子 (1–16) | 同上 |

**技术统计列**包括 FGM/FGA（投篮命中/出手）、FGM3/FGA3（三分）、FTM/FTA（罚球）、OR/DR（进攻/防守篮板）、Ast（助攻）、TO（失误）、Stl（抢断）、Blk（盖帽）、PF（犯规）。

数据截至 2026 年 2 月初。剩余常规赛周将在遴选周日（3 月 15 日）前补充。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_column", 999)
data_dir = "march-machine-learning-mania-2026"

# W -> 女篮, M -> 男篮
M_regular_results = pd.read_csv(f"{data_dir}/MRegularSeasonDetailedResults.csv")
M_tourney_results = pd.read_csv(f"{data_dir}/MNCAATourneyDetailedResults.csv")
M_seeds = pd.read_csv(f"{data_dir}/MNCAATourneySeeds.csv")

W_regular_results = pd.read_csv(f"{data_dir}/WRegularSeasonDetailedResults.csv")
W_tourney_results = pd.read_csv(f"{data_dir}/WNCAATourneyDetailedResults.csv")
W_seeds = pd.read_csv(f"{data_dir}/WNCAATourneySeeds.csv")

我们将男篮和女篮数据合并为单一 DataFrame。男篮 TeamID 以 `1` 开头（范围 1000–1999）；女篮以 `3` 开头（3000–3999）。

**为何共用一个模型？**  
合并使训练集约翻倍，有助于 XGBoost 学习通用锦标赛规律（如「高种子倾向于以 X 分取胜」）。`men_women` 二值标志让模型在存在差异时学习性别相关调整（如女篮的种子爆冷率往往更高）。这比训练两个各自只用半数数据的独立模型更高效。


In [ ]:
# 合并男篮和女篮数据
regular_results = pd.concat([M_regular_results, W_regular_results])
tourney_results = pd.concat([M_tourney_results, W_tourney_results])
seeds = pd.concat([M_seeds, W_seeds])

我们使用 2003 年起的数据——男篮最早具有详细技术统计的赛季。更早赛季仅有简明结果（胜/负 + 比分），无完整技术统计。

> **设计说明**：可将 compact-result 赛季纳入以扩大 Elo/GLM 训练窗口，但技术统计特征在这些年份会为 NaN。当前做法使所有训练行的特征完整性保持一致。


In [ ]:
season = 2003  # 可修改以变更模型截止年份
regular_results = regular_results.loc[regular_results["Season"] >= season]
tourney_results = tourney_results.loc[tourney_results["Season"] >= season]
seeds = seeds.loc[seeds["Season"] >= season]

## <<- 探索数据 ->>


In [ ]:
regular_results

In [ ]:
tourney_results

In [ ]:
# 取一支随机队伍，观察其 2024 赛季的赛程
season = 2024
teamid = 3163

r = regular_results.loc[
    (regular_results["Season"] == season)
    & ((regular_results["WTeamID"] == teamid) | (regular_results["LTeamID"] == teamid))
]
t = tourney_results.loc[
    (tourney_results["Season"] == season)
    & ((tourney_results["WTeamID"] == teamid) | (tourney_results["LTeamID"] == teamid))
]
r["win"] = np.where(r["WTeamID"] == teamid, "win", "lose")
t["win"] = np.where(t["WTeamID"] == teamid, "win", "lose")
r["type"] = "regular season"
t["type"] = "tournament"

rt = pd.concat([r, t])
rt[["DayNum", "WScore", "LScore", "type", "win"]]

In [ ]:
seeds

In [ ]:
# 按分区查看种子结构
s = W_seeds.loc[W_seeds["Season"] == 2015]
[s.loc[s["Seed"].str.startswith(d)] for d in ("X", "Y", "Z", "W")]

In [ ]:
# 查看上一示例队伍的种子
seeds.loc[(seeds["Season"] == season) & (seeds["TeamID"] == teamid)]

## <<- 准备数据 ->>


## 2. 准备数据

**核心设计选择 — 对称翻倍**：每场比赛录入**两次**——一次队伍 A 为 T1、队伍 B 为 T2，一次对调。

*为何？*  
预测时我们收到形如 `2026_1234_1456` 的对阵 ID，其中 TeamID 较小者恒为 T1。模型必须预测 T1 的胜率，**不论哪支队伍实际上更强**。通过在两种顺序上训练，模型在 T1、T2 位置都能看到每支队伍，自然保持对称。否则模型会学到与 TeamID 顺序相关的虚假「T1 优势」。

**加时调整**：技术统计（得分、篮板等）包含加时赛在内的全场累计。打双加时的比赛比常规时间多约 12.5%。我们将所有计数统计归一化到 40 分钟等效：

$$\text{调整后统计} = \text{原始统计} \times \frac{40}{40 + 5 \times \text{NumOT}}$$

避免加时赛在第四节（技术统计场均）中虚增队伍场均。

> **注意**：该调整假设统计随时间线性变化，近似成立但不完全（疲劳、犯规困扰、加时策略变化等）。更复杂的方法可对这些非线性建模。


In [ ]:
# 将数据集对称翻倍（互换队伍位置的技术统计）
def prepare_data(df):
    df = df[
        [
            "Season",
            "DayNum",
            "LTeamID",
            "LScore",
            "WTeamID",
            "WScore",
            "NumOT",
            "LFGM",
            "LFGA",
            "LFGM3",
            "LFGA3",
            "LFTM",
            "LFTA",
            "LOR",
            "LDR",
            "LAst",
            "LTO",
            "LStl",
            "LBlk",
            "LPF",
            "WFGM",
            "WFGA",
            "WFGM3",
            "WFGA3",
            "WFTM",
            "WFTA",
            "WOR",
            "WDR",
            "WAst",
            "WTO",
            "WStl",
            "WBlk",
            "WPF",
        ]
    ]

    # 加时调整因子：加时赛会累积更多统计
    adjot = (40 + 5 * df["NumOT"]) / 40
    adjcols = [
        "LScore",
        "WScore",
        "LFGM",
        "LFGA",
        "LFGM3",
        "LFGA3",
        "LFTM",
        "LFTA",
        "LOR",
        "LDR",
        "LAst",
        "LTO",
        "LStl",
        "LBlk",
        "LPF",
        "WFGM",
        "WFGA",
        "WFGM3",
        "WFGA3",
        "WFTM",
        "WFTA",
        "WOR",
        "WDR",
        "WAst",
        "WTO",
        "WStl",
        "WBlk",
        "WPF",
    ]
    for col in adjcols:
        df[col] = df[col] / adjot

    dfswap = df.copy()
    df.columns = [x.replace("W", "T1_").replace("L", "T2_") for x in list(df.columns)]
    dfswap.columns = [
        x.replace("L", "T1_").replace("W", "T2_") for x in list(dfswap.columns)
    ]
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output["PointDiff"] = output["T1_Score"] - output["T2_Score"]
    output["win"] = (output["PointDiff"] > 0) * 1
    output["men_women"] = (
        output["T1_TeamID"].apply(lambda t: str(t).startswith("1"))
    ) * 1  # 0: 女篮, 1: 男篮
    return output


regular_data = prepare_data(regular_results)
tourney_data = prepare_data(tourney_results)

In [ ]:
regular_data

In [ ]:
tourney_data

In [ ]:
# 取一场示例比赛验证是否在数据集中正确录入
season = regular_data["Season"] == 2025
t1, t2 = 1182, 1433
match1 = (regular_data["T1_TeamID"] == t1) & (regular_data["T2_TeamID"] == t2)
match2 = (regular_data["T1_TeamID"] == t2) & (regular_data["T2_TeamID"] == t1)
regular_data.loc[season & (match1 | match2)]

## <<- 基础难度特征 ->>


## 3. 特征工程 — 基础（种子）

锦标赛种子 (1–16) 由 NCAA 遴选委员会在遴选周日（3 月 15 日）确定。

- **T1_seed, T2_seed**：绝对种子编号（1 最强，16 最弱，各区内部）
- **Seed_diff = T2_seed − T1_seed**：正值表示 T1 为更强种子

**为何种子是强预测因子？**  
种子编码了数月专家判断：委员会考虑胜败记录、赛程强度、直接交锋、联盟锦标赛表现、伤病等。实质上，种子是**人工精选特征**，将可得信息压缩为单一序数排名。

历史上，仅种子即可达到约 74% 的锦标赛结果预测精度。1 对 16 的爆冷率仅约 2.4%（男篮；截至 2024 年 144 场中 1 次爆冷）。

**限制**：种子仅在遴选周日之后可得，故无法用于锦标赛前预测。第 4–6 节的技术统计、Elo、GLM 特征全年提供信号。


In [ ]:
# 从 Seed 字段提取种子编号
seeds["seed"] = seeds["Seed"].apply(lambda x: int(x[1:3]))
seeds

In [ ]:
seeds_T1 = seeds[["Season", "TeamID", "seed"]].copy()
seeds_T2 = seeds[["Season", "TeamID", "seed"]].copy()
seeds_T1.columns = ["Season", "T1_TeamID", "T1_seed"]
seeds_T2.columns = ["Season", "T2_TeamID", "T2_seed"]

tourney_data = tourney_data[
    ["Season", "T1_TeamID", "T2_TeamID", "PointDiff", "win", "men_women"]
]
tourney_data = pd.merge(tourney_data, seeds_T1, on=["Season", "T1_TeamID"], how="left")
tourney_data = pd.merge(tourney_data, seeds_T2, on=["Season", "T2_TeamID"], how="left")
tourney_data["Seed_diff"] = tourney_data["T2_seed"] - tourney_data["T1_seed"]

tourney_data

In [ ]:
# 验证种子是否对分差有预测力
tmpmean = tourney_data.pivot_table(
    columns="men_women", index="T1_seed", values="PointDiff", aggfunc="mean"
).ffill()
tmpstd = tourney_data.pivot_table(
    columns="men_women", index="T1_seed", values="PointDiff", aggfunc="std"
).ffill()
fig, axis = plt.subplots(ncols=2, figsize=(12, 4))
(line_1,) = axis[0].plot(tmpmean.index, tmpmean[0], "b-")
fill_1 = axis[0].fill_between(
    tmpmean.index, tmpmean[0] - tmpstd[0], tmpmean[0] + tmpstd[0], color="b", alpha=0.1
)
(line_2,) = axis[1].plot(tmpmean.index, tmpmean[1], "r--")
fill_2 = axis[1].fill_between(
    tmpmean.index, tmpmean[1] - tmpstd[1], tmpmean[1] + tmpstd[1], color="r", alpha=0.1
)
plt.margins(x=0)
plt.legend([(line_1, fill_1), (line_2, fill_2)], ["女篮", "男篮"])

**上图解读**（种子 → 分差）：

- 横轴为 T1 的种子编号；纵轴为所有锦标赛比赛的平均分差（T1 得分 − T2 得分）。
- **左图（女篮）**：低种子（1–4）平均比对手多约 5–10 分。高种子（13–16）少约 5–10 分。阴影带（±1 标准差）较宽，说明方差大——爆冷常见。
- **右图（男篮）**：趋势类似但带宽略窄，表明男篮锦标赛结果由种子可预测性略高。
- **要点**：种子与分差呈单调关系，确认为强线性预测因子。但标准差较宽意味着仅靠种子仍有很大改进空间。


In [ ]:
# 验证种子差是否对分差有预测力
tmpmean = tourney_data.pivot_table(
    columns="men_women", index="Seed_diff", values="PointDiff", aggfunc="mean"
).ffill()
tmpstd = tourney_data.pivot_table(
    columns="men_women", index="Seed_diff", values="PointDiff", aggfunc="std"
).ffill()
fig, axis = plt.subplots(ncols=2, figsize=(12, 4))
(line_1,) = axis[0].plot(tmpmean.index, tmpmean[0], "b-")
fill_1 = axis[0].fill_between(
    tmpmean.index, tmpmean[0] - tmpstd[0], tmpmean[0] + tmpstd[0], color="b", alpha=0.1
)
(line_2,) = axis[1].plot(tmpmean.index, tmpmean[1], "r--")
fill_2 = axis[1].fill_between(
    tmpmean.index, tmpmean[1] - tmpstd[1], tmpmean[1] + tmpstd[1], color="r", alpha=0.1
)
plt.margins(x=0)
plt.legend([(line_1, fill_1), (line_2, fill_2)], ["女篮", "男篮"])

**上图解读**（种子差 → 分差）：

- **Seed_diff**（横轴）= T2_seed − T1_seed。正值表示 T1 为更好（更低）种子。
- 关系近似**线性**：每单位种子优势约对应 1–1.5 分额外分差。
- 在零附近的对称性可做合理性检验——证实第 2 节的数据集翻倍实现正确。
- 标准差带大致恒定，说明即使种子悬殊，结果方差仍很大（疯狂三月名副其实）。


## <<- 中等难度特征 ->>


In [ ]:
# 技术统计列，用于构建模型特征
boxcols = [
    "T1_Score",
    "T1_FGM",
    "T1_FGA",
    "T1_FGM3",
    "T1_FGA3",
    "T1_FTM",
    "T1_FTA",
    "T1_OR",
    "T1_DR",
    "T1_Ast",
    "T1_TO",
    "T1_Stl",
    "T1_Blk",
    "T1_PF",
    "T2_Score",
    "T2_FGM",
    "T2_FGA",
    "T2_FGM3",
    "T2_FGA3",
    "T2_FTM",
    "T2_FTA",
    "T2_OR",
    "T2_DR",
    "T2_Ast",
    "T2_TO",
    "T2_Stl",
    "T2_Blk",
    "T2_PF",
    "PointDiff",
]

### 直觉：为何需要*同时*有队伍统计与对手统计？

假设队伍 `T` 场均 80 分。这算强吗？取决于对手是谁：
- 若 `T` 的对手对其他人场均 85 分，但对 `T` 仅 70 分 → `T` 防守出色
- 若 `T` 的对手对所有人场均仅 60 分 → `T` 赛程较弱

同时包含 `T` 的自身场均（进攻实力）**和**对手对 `T` 的表现（防守实力/赛程难度），可给模型更完整的图景：

| 特征集 | 刻画内容 |
|------------|-----------------|
| `T1_avg_Score`, `T1_avg_FGA`, ... | T1 的进攻画像 |
| `T1_avg_opponent_Score`, `T1_avg_opponent_FGA`, ... | T1 的防守画像 + 赛程强度 |
| `T2_avg_Score`, `T2_avg_FGA`, ... | T2 的进攻画像 |
| `T2_avg_opponent_Score`, `T2_avg_opponent_FGA`, ... | T2 的防守画像 + 赛程强度 |


## 4. 特征工程 — 中等（技术统计场均）

对每个队伍-赛季，我们计算所有技术统计的**常规赛均值**。`groupby(["Season", "T1_TeamID"]).mean()` 对该赛季该队所有比赛取平均。

因第 2 节对数据集做了翻倍，每队在 T1 位置恰好出现一半行（T2 列为对手统计）。故：
- `T1_avg_Score` = T1 场均得分
- `T1_avg_opponent_Score` = T1 对手**对 T1** 的场均得分（即 T1 的防守质量）

> **为何用简单均值而非加权/近期均值？**  
> 整季均值稳定且方差低。更复杂方法（对近期比赛指数加权、按对手调整）或有帮助但易在小锦标赛训练集上过拟合。本方案保持简单，让 XGBoost 自行发现非线性交互。

> **可改进方向**：用最近 N 场滚动均值捕捉赛季末状态，对锦标赛「火热」队伍尤其重要。


In [ ]:
# 计算赛季场均
ss = regular_data.groupby(["Season", "T1_TeamID"])[boxcols].agg("mean").reset_index()

ss_T1 = ss.copy()
ss_T1.columns = [
    "T1_avg_" + x.replace("T1_", "").replace("T2_", "opponent_")
    for x in list(ss_T1.columns)
]
ss_T1 = ss_T1.rename({"T1_avg_Season": "Season", "T1_avg_TeamID": "T1_TeamID"}, axis=1)
ss_T2 = ss.copy()
ss_T2.columns = [
    "T2_avg_" + x.replace("T1_", "").replace("T2_", "opponent_")
    for x in list(ss_T2.columns)
]
ss_T2 = ss_T2.rename({"T2_avg_Season": "Season", "T2_avg_TeamID": "T2_TeamID"}, axis=1)

tourney_data = pd.merge(tourney_data, ss_T1, on=["Season", "T1_TeamID"], how="left")
tourney_data = pd.merge(tourney_data, ss_T2, on=["Season", "T2_TeamID"], how="left")
tourney_data

## <<- 复杂难度特征 ->>


## 5. 特征工程 — 复杂（Elo 评级）

### 背景

**Elo 评级系统**由物理学家 Arpad Elo 为国际象棋排名发明。核心思想：每队有数字强度评级，每场比赛后胜者加分、负者减分，变动量取决于赛果是否符合预期。

### 数学原理

队伍 A（评级 $R_A$）与队伍 B（评级 $R_B$）比赛前，A 的**预期得分**为：

$$E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}$$

这是以 $R_A = R_B$（此时 $E_A = 0.5$）为中心的逻辑函数。分母 400 控制预期随评级差的变化速度：
- 400 分差 → $E_A \approx 0.91$（明显占优）
- 200 分差 → $E_A \approx 0.76$
- 0 分差 → $E_A = 0.50$（五五开）

赛后评级更新：

$$R_A^{\text{new}} = R_A + K \cdot (S_A - E_A)$$

其中 $S_A = 1$ 若 A 胜，$0$ 若 A 负，$K$ 为控制更新幅度的 **K 因子**。

### 本 notebook 中的实现选择

| 参数 | 取值 | 说明 |
|-----------|-------|-----------|
| `base_elo` | 1000 | 每赛季所有队伍起始评级 |
| `elo_width` | 400 | 标准 Elo 尺度 |
| `k_factor` | 100 | **非常激进**——单场爆冷可使评级波动约 50 分。大学篮球常规赛仅约 30 场，每场比赛权重大，这合理。（对比：国际象棋常用 K=10–40。） |

**赛季重置**：每赛季所有队伍从 1000 重新开始。虽丢弃跨季信息，但避免阵容变动导致上季 Elo 失效。

> **可改进方向**：  
> - **分差加权 Elo**：按胜分差缩放 K 因子，区分大胜与险胜  
> - **跨季延续**：用如 `0.75 × 上季_elo + 0.25 × 1000` 初始化以反映留队球员  
> - **主场调整**：从主队预期得分中减去固定值（NCAA 数据含 `WLoc`）


In [ ]:
def update_elo(winner_elo, loser_elo):
    expected_win = expected_result(winner_elo, loser_elo)
    change_in_elo = k_factor * (1 - expected_win)
    winner_elo += change_in_elo
    loser_elo -= change_in_elo
    return winner_elo, loser_elo


def expected_result(elo_a, elo_b):
    return 1.0 / (1 + 10 ** ((elo_b - elo_a) / elo_width))


base_elo = 1000
elo_width = 400
k_factor = 100

elos = []
for season in sorted(set(seeds["Season"])):
    ss = regular_data.loc[regular_data["Season"] == season]
    ss = ss.loc[ss["win"] == 1].reset_index(drop=True)
    teams = set(ss["T1_TeamID"]) | set(ss["T2_TeamID"])
    elo = dict(zip(teams, [base_elo] * len(teams)))
    for i in range(ss.shape[0]):
        w_team, l_team = ss.loc[i, "T1_TeamID"], ss.loc[i, "T2_TeamID"]
        w_elo, l_elo = elo[w_team], elo[l_team]
        w_elo_new, l_elo_new = update_elo(w_elo, l_elo)
        elo[w_team] = w_elo_new
        elo[l_team] = l_elo_new
    elo = pd.DataFrame.from_dict(elo, orient="index").reset_index()
    elo = elo.rename({"index": "TeamID", 0: "elo"}, axis=1)
    elo["Season"] = season
    elos.append(elo)
elos = pd.concat(elos)

elos_T1 = elos.copy().rename({"TeamID": "T1_TeamID", "elo": "T1_elo"}, axis=1)
elos_T2 = elos.copy().rename({"TeamID": "T2_TeamID", "elo": "T2_elo"}, axis=1)
tourney_data = pd.merge(tourney_data, elos_T1, on=["Season", "T1_TeamID"], how="left")
tourney_data = pd.merge(tourney_data, elos_T2, on=["Season", "T2_TeamID"], how="left")
tourney_data["elo_diff"] = tourney_data["T1_elo"] - tourney_data["T2_elo"]

In [ ]:
tmp = pd.merge(
    elos,
    tourney_data[["Season", "T1_TeamID"]].drop_duplicates(),
    left_on=["Season", "TeamID"],
    right_on=["Season", "T1_TeamID"],
    how="left",
)

plt.figure(figsize=(6, 4))
seaborn.distplot(tmp.loc[pd.isnull(tmp["T1_TeamID"]), "elo"], kde=False)
seaborn.distplot(tmp.loc[~pd.isnull(tmp["T1_TeamID"]), "elo"], kde=False)

**上图解读**（Elo 分布）：

该直方图比较赛季末 Elo 评级：
- **未进锦标赛的队伍**（左/蓝分布）：集中在 1000 附近，符合预期——多数队伍接近起始评级
- **进锦标赛的队伍**（右/橙分布）：整体右移，中心约在 1100–1200

**分布分离**表明 Elo 具有信息量：锦标赛队伍普遍更高。但重叠仍大——部分锦标赛队伍 Elo 一般（如弱联盟自动晋级），部分非锦标赛队伍 Elo 较高（如落选边缘队）。


In [ ]:
plt.figure(figsize=(6, 4))
seaborn.stripplot(data=tourney_data, y="T1_elo", x="T1_seed", size=3)

**上图解读**（Elo vs 种子）：

每点代表锦标赛中一支队伍-赛季。横轴为种子 (1–16)，纵轴为赛季末 Elo。

- **强负相关**：1 号种子聚集在 Elo ~1300+，16 号种子在 1000–1050 附近
- **同种子内方差可观**：5 号种子 Elo 可从 1050 到 1250。Elo 在此处超越种子——可区分同种子队伍。
- 关系非完全线性；在极端种子处被压缩（1 和 16 号种子的 Elo 方差小于中间种子）。


In [ ]:
plt.figure(figsize=(12, 4))
seaborn.stripplot(
    data=tourney_data, y="elo_diff", x="Seed_diff", hue="win", dodge=True, size=3
)
seaborn.lineplot([0] * 29, color="gray", lw=1)

**上图解读**（Elo 差 × 种子差 × 胜/负）：

- 横轴：种子差。纵轴：Elo 差。色调：胜（橙）vs 负（蓝）。
- **当 Elo 与种子一致时**（右上象限：种子差正、Elo 差正），胜者占优。
- **当二者不一致时**（如 T1 种子更好但 Elo 更低），结果混杂→最不确定的比赛，也是模型在 Brier 上拉开差距之处。
- 灰色线 y=0 将 Elo 偏向 T1（上方）与 Elo 偏向 T2（下方）分开。


## <<- 最难难度特征 ->>


## 6. 特征工程 — 最难（GLM 队伍质量）

### 思路

Elo 逐场顺序更新，而 GLM 从所有常规赛**同时**估计队伍实力。拟合高斯广义线性模型：

$$\text{PointDiff}_{ij} = \alpha_i - \alpha_j + \epsilon_{ij}$$

其中 $\alpha_i$ 为队伍 $i$ 的质量系数，$\alpha_j$ 为队伍 $j$ 的。截距设为 0（公式中 `-1`），使系数可解释为「高于/低于平均的分数」。

### 与 Bradley-Terry 的关系

与成对比较中的 **Bradley-Terry 模型**密切相关：

$$P(i \text{ beats } j) = \frac{e^{\alpha_i}}{e^{\alpha_i} + e^{\alpha_j}}$$

关键区别：Bradley-Terry 建模二值结果，而本 GLM 通过高斯似然建模**连续分差**。回归方法每场提取更多信号，因使用胜分差而非仅胜负。

### 实现细节

1. **队伍过滤**：仅包含锦标赛晋级队（来自种子）及至少击败过他们一次的队伍。其余合并为「队伍 0000」。避免模型在无关队伍上浪费自由度，同时仍考虑质量型失利。

2. **独热编码**：`T1_TeamID` 与 `T2_TeamID` 作类别变量，生成大稀疏设计矩阵。每行恰好两个非零（T1 为 +1，T2 通过 GLM 结构隐式为 −1）。

3. **分赛季、分性别拟合**：对每个（赛季，性别）对独立拟合，系数反映当年队伍实力。

4. **解读**：quality = +5.2 表示该队平均比调整后对手多 5.2 分/场。类似「胜场贡献值」指标。

> **为何 GLM 质量优于种子？**  
> 种子将队伍离散为 16 档，且受非表现因素影响（联盟政治、委员会主观性）。GLM 质量为连续、纯数据驱动指标。如下所示，GLM 质量在多数赛季的 AUC 高于种子。

> **可改进方向**：  
> - 在 GLM 公式中加入**主场指示变量**  
> - 用**正则回归**（Ridge/Lasso）替代普通 OLS-GLM 处理多重共线性  
> - 用**滚动窗口**最近比赛拟合，而非整季，以捕捉赛季末状态


In [ ]:
import statsmodels.api as sm
import tqdm

regular_data["ST1"] = regular_data.apply(
    lambda t: str(int(t["Season"])) + "/" + str(int(t["T1_TeamID"])), axis=1
)
regular_data["ST2"] = regular_data.apply(
    lambda t: str(int(t["Season"])) + "/" + str(int(t["T2_TeamID"])), axis=1
)
seeds_T1["ST1"] = seeds_T1.apply(
    lambda t: str(int(t["Season"])) + "/" + str(int(t["T1_TeamID"])), axis=1
)
seeds_T2["ST2"] = seeds_T2.apply(
    lambda t: str(int(t["Season"])) + "/" + str(int(t["T2_TeamID"])), axis=1
)

# 收集锦标赛队伍
st = set(seeds_T1["ST1"]) | set(seeds_T2["ST2"])
# 添加至少击败过锦标赛队伍一次的非锦标赛队伍
st = st | set(
    regular_data.loc[
        (regular_data["T1_Score"] > regular_data["T2_Score"])
        & (regular_data["ST2"].isin(st)),
        "ST1",
    ]
)


def team_quality(season, men_women):
    # 混合效应：固定截距=0，随机斜率
    formula = "PointDiff~-1+T1_TeamID+T2_TeamID"
    glm = sm.GLM.from_formula(
        formula=formula,
        data=dt.loc[(dt["Season"] == season) & (dt["men_women"] == men_women), :],
        family=sm.families.Gaussian(),
    ).fit()

    quality = pd.DataFrame(glm.params).reset_index()
    quality.columns = ["TeamID", "quality"]
    quality["quality"] = quality["quality"]
    quality["Season"] = season
    quality = quality.loc[quality.TeamID.str.contains("T1_")].reset_index(drop=True)
    quality["TeamID"] = quality["TeamID"].apply(lambda x: x[10:14]).astype(int)
    return quality


glm_quality = []

dt = regular_data.loc[regular_data["ST1"].isin(st) | regular_data["ST2"].isin(st)]
dt["T1_TeamID"] = dt["T1_TeamID"].astype(str)
dt["T2_TeamID"] = dt["T2_TeamID"].astype(str)
dt.loc[~dt["ST1"].isin(st), "T1_TeamID"] = "0000"
dt.loc[~dt["ST2"].isin(st), "T2_TeamID"] = "0000"
seasons = sorted(set(seeds["Season"]))
for s in tqdm.tqdm(seasons, unit="season"):
    if s >= 2010:  # 女篮最早赛季
        glm_quality.append(team_quality(s, 0))
    if s >= 2003:  # 男篮最早赛季
        glm_quality.append(team_quality(s, 1))

glm_quality = pd.concat(glm_quality).reset_index(drop=True)

glm_quality_T1 = glm_quality.copy()
glm_quality_T2 = glm_quality.copy()
glm_quality_T1.columns = ["T1_TeamID", "T1_quality", "Season"]
glm_quality_T2.columns = ["T2_TeamID", "T2_quality", "Season"]
tourney_data = pd.merge(
    tourney_data, glm_quality_T1, on=["Season", "T1_TeamID"], how="left"
)
tourney_data = pd.merge(
    tourney_data, glm_quality_T2, on=["Season", "T2_TeamID"], how="left"
)
tourney_data["diff_quality"] = tourney_data["T1_quality"] - tourney_data["T2_quality"]

In [ ]:
tmp = (
    tourney_data[["Season", "men_women", "T1_seed", "T1_quality"]]
    .drop_duplicates()
    .sort_values("T1_quality")
    .reset_index(drop=True)
)

fig, axs = plt.subplots(ncols=2, figsize=(12, 4))

seaborn.lineplot(
    tmp.loc[tmp["men_women"] == 0, "T1_quality"], color="lightgray", ax=axs[0]
)
seaborn.scatterplot(
    tmp.loc[(tmp["men_women"] == 0) & (tmp.T1_seed == 1), "T1_quality"],
    color="red",
    ax=axs[0],
)
seaborn.scatterplot(
    tmp.loc[(tmp["men_women"] == 0) & (tmp.T1_seed == 7), "T1_quality"],
    color="blue",
    ax=axs[0],
)
seaborn.scatterplot(
    tmp.loc[(tmp["men_women"] == 0) & (tmp.T1_seed == 16), "T1_quality"],
    color="green",
    ax=axs[0],
)

seaborn.lineplot(
    tmp.loc[tmp["men_women"] == 1, "T1_quality"], color="lightgray", ax=axs[1]
)
seaborn.scatterplot(
    tmp.loc[(tmp["men_women"] == 1) & (tmp.T1_seed == 1), "T1_quality"],
    color="red",
    ax=axs[1],
)
seaborn.scatterplot(
    tmp.loc[(tmp["men_women"] == 1) & (tmp.T1_seed == 7), "T1_quality"],
    color="blue",
    ax=axs[1],
)
seaborn.scatterplot(
    tmp.loc[(tmp["men_women"] == 1) & (tmp.T1_seed == 16), "T1_quality"],
    color="green",
    ax=axs[1],
)

**上图解读**（GLM 质量 vs 种子）：

- 每块（女篮/男篮）展示各赛季所有锦标赛队伍，按 GLM 质量（横轴）排序
- **红点** = 1 号种子，**蓝** = 7 号种子，**绿** = 16 号种子，灰线 = 全体队伍
- 1 号种子（红）稳定占据最右（最高质量），说明 GLM 与专家排名大体一致
- 但存在**倒挂**：部分 7 号或更低种子队伍 GLM 质量高于同年更高种子。这些错配正是 GLM 特征超越种子的价值所在。
- 整体曲线近似线性带轻微 S 形——极端处质量差异更大。


In [ ]:
tmp["QualitySeed"] = (
    (tmp.groupby(["Season", "men_women"])["T1_quality"].rank(ascending=False) // 4 + 1)
    .clip(1, 16)
    .astype(int)
)
pd.pivot_table(
    data=tmp,
    index="T1_seed",
    columns="QualitySeed",
    values="men_women",
    aggfunc="count",
).fillna(0).astype(int).style.bar(color="#5fba7d", vmin=0, vmax=50)

**Reading the table above** (Seed × Quality-Seed confusion matrix):

`QualitySeed` re-ranks teams by GLM quality into 16 buckets (1 = highest quality). If seeds and GLM agreed perfectly, all counts would lie on the diagonal.

- **Diagonal concentration**: Most mass is near the diagonal, confirming strong agreement.
- **Off-diagonal entries**: These are the "disagreements" — e.g., a team seeded 5 by the committee but ranked 2 by GLM. The model can exploit these disagreements to outperform a seed-only baseline.
- **Lower-right corner** tends to be sparser: the committee rarely seeds a genuinely strong team (high GLM quality) as 15 or 16.


In [ ]:
from sklearn.metrics import roc_auc_score

print(
    "Seed AUC    :",
    np.round(
        roc_auc_score(
            1 - tourney_data["win"], tourney_data["T1_seed"] - tourney_data["T2_seed"]
        ),
        3,
    ),
)
print(
    "Quality AUC :",
    np.round(
        roc_auc_score(
            tourney_data["win"], tourney_data["T1_quality"] - tourney_data["T2_quality"]
        ),
        3,
    ),
)

**Interpreting the AUC comparison above**:

- **Seed AUC ~0.82**: Using only seed difference to predict wins achieves strong discrimination.
- **Quality AUC ~0.84**: GLM quality difference slightly exceeds seeds.
- The gap is small but consistent, and combined they provide complementary signal to the XGBoost model.

> **AUC vs Brier 说明**：AUC 衡量排序质量（能否把胜排在负之上？），而 Kaggle 评估 **Brier 分数**（概率是否校准良好？）。概率未校准时模型可高 AUC 却低 Brier。故第 8 节（样条校准）至关重要。


In [ ]:
# 按赛季比较：专家（种子）vs 统计（GLM）谁更准
for s in sorted(set(tourney_data["Season"])):
    st = tourney_data["Season"] == s
    print(
        s,
        a := np.round(
            roc_auc_score(
                1 - tourney_data.loc[st, "win"],
                tourney_data.loc[st, "T1_seed"] - tourney_data.loc[st, "T2_seed"],
            ),
            3,
        ),
        b := np.round(
            roc_auc_score(
                tourney_data.loc[st, "win"],
                tourney_data.loc[st, "T1_quality"] - tourney_data.loc[st, "T2_quality"],
            ),
            3,
        ),
        np.where(a > b, "", "Q"),
    )

## <<- 机器学习模型 ->>


## 7. 模型 — XGBoost

### 为何对分差做回归而非二分类？

胜率问题看似应做二分类（胜=1，负=0）。但**对分差做回归**有重要优势：

1. **更丰富信号**：20 分大胜与 1 分险胜在分类中都是「胜」。回归能区分统治力差异。
2. **更平滑概率面**：分差→概率转换后（第 8 节）自然得到良好校准概率。XGBoost 二分类易过度自信（概率接近 0 或 1）。
3. **更有利于 Brier 分数**：回归得到的平滑概率通常校准更好，直接提升 Brier 分数。

### 使用的特征

特征列表经过精心筛选——代码中多处可用特征被**注释掉**，这是刻意为之：


In [ ]:
features = [
    ### EASY FEATURES ###
    "men_women",
    "T1_seed",
    "T2_seed",
    "Seed_diff",
    ### MEDIUM FEATURES ###
    "T1_avg_Score",
    # "T1_avg_FGM",
    "T1_avg_FGA",
    # "T1_avg_FGM3",
    # "T1_avg_FGA3",
    # "T1_avg_FTM",
    # "T1_avg_FTA",
    "T1_avg_OR",
    "T1_avg_DR",
    # "T1_avg_Ast",
    # "T1_avg_TO",
    # "T1_avg_Stl",
    "T1_avg_Blk",
    "T1_avg_PF",
    # "T1_avg_opponent_Score",
    # "T1_avg_opponent_FGM",
    "T1_avg_opponent_FGA",
    # "T1_avg_opponent_FGM3",
    # "T1_avg_opponent_FGA3",
    # "T1_avg_opponent_FTM",
    # "T1_avg_opponent_FTA",
    # "T1_avg_opponent_OR",
    # "T1_avg_opponent_DR",
    # "T1_avg_opponent_Ast",
    # "T1_avg_opponent_TO",
    # "T1_avg_opponent_Stl",
    "T1_avg_opponent_Blk",
    "T1_avg_opponent_PF",
    "T1_avg_PointDiff",
    "T2_avg_Score",
    # "T2_avg_FGM",
    "T2_avg_FGA",
    # "T2_avg_FGM3",
    # "T2_avg_FGA3",
    # "T2_avg_FTM",
    # "T2_avg_FTA",
    "T2_avg_OR",
    "T2_avg_DR",
    # "T2_avg_Ast",
    # "T2_avg_TO",
    # "T2_avg_Stl",
    "T2_avg_Blk",
    "T2_avg_PF",
    # "T2_avg_opponent_Score",
    # "T2_avg_opponent_FGM",
    "T2_avg_opponent_FGA",
    # "T2_avg_opponent_FGM3",
    # "T2_avg_opponent_FGA3",
    # "T2_avg_opponent_FTM",
    # "T2_avg_opponent_FTA",
    # "T2_avg_opponent_OR",
    # "T2_avg_opponent_DR",
    # "T2_avg_opponent_Ast",
    # "T2_avg_opponent_TO",
    # "T2_avg_opponent_Stl",
    "T2_avg_opponent_Blk",
    "T2_avg_opponent_PF",
    "T2_avg_PointDiff",
    ### HARD FEATURES ###
    "T1_elo",
    "T2_elo",
    "elo_diff",
    ### HARDEST FEATURES ###
    "T1_quality",
    "T2_quality",
]

print(f"特征数量 {len(features)}")

### 特征选择理由

**纳入特征**及理由：

| 类别 | 特征 | 纳入理由 |
|----------|----------|-------------|
| 种子 | `T1_seed`, `T2_seed`, `Seed_diff` | 最强单一预测因子；编码专家判断 |
| 得分 | `avg_Score`, `avg_FGA` | 核心进攻输出；FGA 与节奏相关 |
| 篮板 | `avg_OR`, `avg_DR` | OR=二次进攻；DR=防守回合 |
| 防守 | `avg_Blk`, `avg_PF`, `avg_opponent_FGA/Blk/PF` | 盖帽反映护筐；犯规反映侵略性 |
| Elo | `T1_elo`, `T2_elo`, `elo_diff` | 全季累计实力信号 |
| 质量 | `T1_quality`, `T2_quality` | GLM 同时估计队伍实力 |

**排除特征**（已注释）及理由：

- `FGM`（投篮命中）：与 `Score`、`FGA` 冗余——已知出手和得分可推知命中
- `FGM3`/`FGA3`：三分单场方差大，赛季均值为噪声
- `FTM`/`FTA`：罚球更多反映造犯规风格而非队伍质量
- `Ast`, `TO`, `Stl`：单看有信息量，但与已选特征多重共线性；锦标赛数据有限时，XGBoost 易因特征过多过拟合噪声

> **设计原则**：约 20 赛季仅 ~3000 场锦标赛（翻倍后 ~6000），过拟合风险真实存在。保持特征集精简（27 个）有助于泛化。注释掉的特征便于实验。


### XGBoost 超参数（在历史数据上调优）

| 参数 | 取值 | 作用 |
|-----------|-------|-------------|
| `eta` | 0.0093 | 学习率——极低，需多轮但收敛稳定 |
| `num_boost_round` | 704 | 提升迭代数；配合低 eta 逐步学习 |
| `subsample` | 0.6 | 行采样——每棵树见 60% 数据，减轻过拟合 |
| `colsample_bynode` | 0.8 | 每分裂节点特征采样——增加树间多样性 |
| `num_parallel_tree` | 2 | 每轮训练 2 棵树（每步小随机森林）——平滑预测 |
| `min_child_weight` | 4 | 叶节点最小样本权重和——防止在极小群体上分裂 |
| `max_depth` | 4 | 浅树；深树会记忆具体对阵 |
| `max_bin` | 38 | 直方图分箱数——略高于默认以获更细分裂 |
| `grow_policy` | `lossguide` | 叶向生长（类似 LightGBM）——分裂更高效 |

> **为何这些取值？** 原作者 (modeh7)  很可能通过历史锦标赛交叉验证调优。极低学习率、多轮与强正则（subsample、colsample、浅深度）是小数据集**稳定泛化梯度提升**的经典配置。


In [ ]:
import xgboost as xgb

param = {}
param["objective"] = "reg:squarederror"
param["booster"] = "gbtree"
param["eta"] = 0.0093  # 0.009 #0.01
param["subsample"] = 0.6
param["colsample_bynode"] = 0.8
param["num_parallel_tree"] = 2
param["min_child_weight"] = 4
param["max_depth"] = 4
param["tree_method"] = "hist"
param["grow_policy"] = "lossguide"
param["max_bin"] = 38  # 32

num_rounds = 704  # 704 #700

### 验证：留一赛季法 (LOSO)

对数据集中每个赛季 $s$：
1. **Train** on all tournament games from seasons $\neq s$
2. **Predict** season $s$'s tournament games
3. Record out-of-fold predictions

**为何用 LOSO 而非随机 K 折？**

- **时间真实性**：实际用过去赛季训练、预测未来。LOSO 完全模仿该流程。随机 K 折会把未来信息泄露进训练。
- **赛季级相关性**：同季比赛共享队伍、Elo 轨迹和 GLM 系数。按赛季划分可防止该相关性虚高验证分数。
- **每折一模型 = 推理时集成**：各 LOSO 模型训练分布略有不同。推理时平均约 20 个模型预测等价于免费**模型集成**，通常降低方差并使 Brier 提升 1–3%。

> **微妙泄露风险**：Elo 和 GLM 特征仅从常规赛计算（不含锦标赛），故无目标泄露。但样条校准（第 8 节）在所有 OOF 预测合并后拟合，即每季校准曲线受所有赛季影响。更严格做法是在嵌套 CV 中拟合样条，但鉴于样条平滑性，影响可能较小。


In [ ]:
from sklearn.metrics import mean_absolute_error, brier_score_loss

models = {}
oof_mae = []
oof_preds = []
oof_targets = []
oof_ss = []

# leave-one-season out models
for oof_season in set(tourney_data.Season):
    x_train = tourney_data.loc[tourney_data["Season"] != oof_season, features].values
    y_train = tourney_data.loc[tourney_data["Season"] != oof_season, "PointDiff"].values
    x_val = tourney_data.loc[tourney_data["Season"] == oof_season, features].values
    y_val = tourney_data.loc[tourney_data["Season"] == oof_season, "PointDiff"].values
    s_val = tourney_data.loc[tourney_data["Season"] == oof_season, "Season"].values

    dtrain = xgb.DMatrix(x_train, label=y_train)
    dval = xgb.DMatrix(x_val, label=y_val)
    models[oof_season] = xgb.train(
        params=param,
        dtrain=dtrain,
        num_boost_round=num_rounds,
    )
    preds = models[oof_season].predict(dval)
    print(f"oof season {oof_season} mae: {mean_absolute_error(y_val, preds)}")
    oof_mae.append(mean_absolute_error(y_val, preds))
    oof_preds += list(preds)
    oof_targets += list(y_val)
    oof_ss += list(s_val)

print(f"average mae: {np.mean(oof_mae)}")

In [ ]:
df = pd.DataFrame(
    {
        "Season": oof_ss,
        "pred": oof_preds,
        "label": [(t > 0) * 1 for t in oof_targets],
        "men_women": tourney_data["men_women"],
    }
)
df["pred_pointdiff"] = df["pred"].astype(int)

xdf_all = (
    df.clip(-30, 30)
    .groupby("pred_pointdiff")["label"]
    .mean()
    .reset_index(name="average_win_pct")
)
xdf_men = (
    df.clip(-30, 30)
    .loc[df["men_women"] == 0]
    .groupby("pred_pointdiff")["label"]
    .mean()
    .reset_index(name="average_win_pct")
)
xdf_women = (
    df.clip(-30, 30)
    .loc[df["men_women"] == 1]
    .groupby("pred_pointdiff")["label"]
    .mean()
    .reset_index(name="average_win_pct")
)

seaborn.lineplot(x=xdf_all["pred_pointdiff"], y=xdf_all["average_win_pct"])
seaborn.lineplot(x=xdf_men["pred_pointdiff"], y=xdf_men["average_win_pct"])
seaborn.lineplot(x=xdf_women["pred_pointdiff"], y=xdf_women["average_win_pct"])

**Reading the plot above** (Predicted Point Diff → Empirical Win Rate):

- The three lines show empirical win rate as a function of predicted point differential (binned to integers), for all games, men only, and women only.
- **Ideal shape**: A monotonically increasing sigmoid-like curve from ~0% (large negative diff) to ~100% (large positive diff), crossing 50% near zero.
- The curves are generally well-behaved but **noisy at the extremes** (few games with predicted margin > ±20).
- If the men's and women's curves differ substantially, it suggests the model might benefit from separate calibration per gender. Here they track closely, supporting the unified approach.


## 8. 分差 → 概率（样条校准）

### 问题

XGBoost outputs a continuous point differential prediction (e.g., +7.3 means "T1 wins by ~7 points"). But Kaggle requires a **probability** in [0, 1]. We need a calibration function:

$$p = f(\hat{d})$$

where $\hat{d}$ is the predicted point diff and $p$ is the win probability.

### 为何用样条而非 logistic 函数？

A natural choice would be logistic regression: $p = 1/(1 + e^{-\beta \hat{d}})$. But this assumes a specific symmetric functional form. A **quintic spline** ($k=5$) is more flexible:

- It's fit on OOF predictions: for each predicted point diff, we know the actual binary outcome.
- The spline finds a smooth curve through the (predicted diff, empirical win rate) scatter.
- Degree 5 allows sufficient curvature to capture any asymmetry (e.g., maybe +10 in women's games maps to a different win probability than +10 in men's games, though here they're combined).

### 在 ±25 处截断

超过 ±25 分的预测在样条评估前被截断。避免样条在数据稀疏区（真实分差 > 25 的比赛极少）外推失真。

### Brier 分数的解读

**Brier score** = mean squared error between predicted probabilities and binary outcomes:

$$\text{Brier} = \frac{1}{N}\sum_{i=1}^{N}(p_i - y_i)^2$$

越低越好。大学篮球 Brier 分数约 0.20 为常见水平（可与 FiveThirtyEight 模型相比）。Brier 可分解为**校准**（概率是否准确？）与**分辨率**（能否区分胜负？）。样条主要提升校准部分。

> **可改进方向**：为男篮与女篮锦标赛分别拟合样条，或采用**保序回归**作为备选非参数校准器（保证单调性，而样条不严格保证）。


In [ ]:
from scipy.interpolate import UnivariateSpline

t = 25
dat = list(zip(oof_preds, np.array(oof_targets) > 0))
dat = sorted(dat, key=lambda x: x[0])
pred, label = list(zip(*dat))
spline_model = UnivariateSpline(np.clip(pred, -t, t), label, k=5)
spline_fit = np.clip(spline_model(np.clip(oof_preds, -t, t)), 0.01, 0.99)
print(f"brier: {brier_score_loss(np.array(oof_targets)>0, spline_fit)}")
df["spline"] = spline_fit
xdf = (
    df.clip(-30, 30).groupby("pred_pointdiff")[["spline", "label"]].mean().reset_index()
)

plt.figure()
plt.plot(xdf["pred_pointdiff"], xdf["label"])
plt.plot(xdf["pred_pointdiff"], xdf["spline"])

**Reading the plot above** (Spline Calibration Curve):

- **Blue line**: Empirical win rate at each predicted point-diff bin (the "truth" we want to match)
- **Orange line**: Spline-fitted probability

样条应紧贴经验曲线。两条线间的空隙表示校准误差。在极端处（|diff| > 20），两曲线因数据有限而噪声增大。

A well-calibrated model means: among all games where the model predicts 70% win probability, approximately 70% are actual wins.


In [ ]:
print(f"brier: {brier_score_loss(np.array(oof_targets)>0, spline_fit)}")

for oof_season in set(tourney_data.Season):
    x = df.loc[df["Season"] == oof_season, "spline"].values
    y = df.loc[df["Season"] == oof_season, "label"].values
    print(oof_season, np.round(brier_score_loss(y, x), 5))

## <<- 生成提交 ->>


In [ ]:
X = pd.read_csv(f"{data_dir}/SampleSubmissionStage1.csv")
X

## 9. 生成提交

对 Stage 2 的每个可能对阵，我们：
1. **合并**所有特征：技术统计场均、种子、Elo、GLM 质量
2. **集成预测**：用约 20 个 LOSO 模型对同一输入预测，取分差预测均值
3. **校准**：通过样条将平均分差转为概率
4. **截断**至 [0.01, 0.99]，限制过度自信错误预测的惩罚（对 Brier 不如 log-loss 敏感，但仍为良好实践）

`models[oof_season].predict(dtest) * 1.0` 中的乘数可调节激进程度：>1 使预测更远离 0.5（更自信），<1 使其更靠近 0.5（更保守）。当前为 1.0（中性）。

> **重要**：2026 年种子在遴选周日（3 月 15 日）前均为 NaN。在此之前运行会使种子相关特征为 NaN，XGBoost 会将其作为缺失值处理（使用其原生缺失值支持）。种子公布后需重新运行以生成最终提交。

> **关于已移除的手动覆写**：2025 方案曾对特定对阵基于领域知识硬编码概率覆写。2026 因对阵尚未确定已移除。bracket 确定后，手动调整（如针对关键球员伤病）可提升特定场次的预测表现。


In [ ]:
# construct dataframe for submission
X["Season"] = X["ID"].apply(lambda t: int(t.split("_")[0]))
X["T1_TeamID"] = X["ID"].apply(lambda t: int(t.split("_")[1]))
X["T2_TeamID"] = X["ID"].apply(lambda t: int(t.split("_")[2]))
X["men_women"] = X["T1_TeamID"].apply(lambda t: 1 if str(t)[0] == "1" else 0)
X = pd.merge(X, ss_T1, on=["Season", "T1_TeamID"], how="left")
X = pd.merge(X, ss_T2, on=["Season", "T2_TeamID"], how="left")
X = pd.merge(X, seeds_T1, on=["Season", "T1_TeamID"], how="left")
X = pd.merge(X, seeds_T2, on=["Season", "T2_TeamID"], how="left")
X = pd.merge(X, glm_quality_T1, on=["Season", "T1_TeamID"], how="left")
X = pd.merge(X, glm_quality_T2, on=["Season", "T2_TeamID"], how="left")
X = pd.merge(X, elos_T1, on=["Season", "T1_TeamID"], how="left")
X = pd.merge(X, elos_T2, on=["Season", "T2_TeamID"], how="left")
X["Seed_diff"] = X["T2_seed"] - X["T1_seed"]
X["elo_diff"] = X["T1_elo"] - X["T2_elo"]
X["diff_quality"] = X["T1_quality"] - X["T2_quality"]

In [ ]:
# run models on given dataset
preds = []
for oof_season in set(tourney_data.Season):
    dtest = xgb.DMatrix(X[features].values)
    margin_preds = (
        models[oof_season].predict(dtest) * 1.0
    )  # aggressive submissions >1, conservative submissions <1
    probs = np.clip(spline_model(np.clip(margin_preds, -t, t)), 0.01, 0.99)
    preds.append(probs)
X["Pred"] = np.array(preds).mean(axis=0)

In [ ]:
# Manual overrides removed for 2026 (matchups not yet known)
# X["Pred"] is used as-is from the model
X["Pred"] = X["Pred"].round(6)

In [ ]:
X[["ID", "Pred"]].to_csv("predictions.csv", index=None)

In [ ]:
print(X["Pred"].mean())
print(X["Pred"].describe())